In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# การจำลองแบบแม่นยำและแบบมีสัญญาณรบกวนด้วย Qiskit Aer primitives

<details>
<summary><b>เวอร์ชันของแพ็กเกจ</b></summary>

โค้ดในหน้านี้พัฒนาด้วยข้อกำหนดต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.3.0
qiskit-aer~=0.17
```
</details>

[การจำลองแบบแม่นยำด้วย Qiskit primitives](/guides/simulate-with-qiskit-sdk-primitives) แสดงวิธีใช้ reference primitives ที่มาพร้อมกับ Qiskit เพื่อจำลอง quantum circuit อย่างแม่นยำ ในปัจจุบัน quantum processor ที่มีอยู่ยังประสบปัญหาข้อผิดพลาดหรือสัญญาณรบกวน ดังนั้นผลลัพธ์จากการจำลองแบบแม่นยำจึงไม่จำเป็นต้องสะท้อนผลลัพธ์ที่คุณจะได้เมื่อรัน circuit บนฮาร์ดแวร์จริง แม้ว่า reference primitives ใน Qiskit จะไม่รองรับการสร้างแบบจำลองสัญญาณรบกวน แต่ [Qiskit Aer](https://qiskit.org/ecosystem/aer/) มีการใช้งาน primitives ที่รองรับการสร้างแบบจำลองสัญญาณรบกวน Qiskit Aer เป็น quantum circuit simulator ประสิทธิภาพสูงที่คุณสามารถใช้แทน reference primitives เพื่อประสิทธิภาพที่ดีขึ้นและฟีเจอร์ที่มากขึ้น โดยเป็นส่วนหนึ่งของ [Qiskit Ecosystem](https://qiskit.github.io/ecosystem/) ในบทความนี้ เราจะสาธิตการใช้ Qiskit Aer primitives สำหรับการจำลองแบบแม่นยำและแบบมีสัญญาณรบกวน

> **Note:** - ต้องใช้ `qiskit-aer` v0.14 หรือใหม่กว่า
> - แม้ว่า Qiskit Aer primitives จะใช้งาน primitive interfaces แต่ก็ไม่ได้ให้ตัวเลือกเดียวกับ Qiskit Runtime primitives เช่น Resilience level ไม่มีให้ใช้งานใน Qiskit Aer primitives
> - ดูรายละเอียดเกี่ยวกับตัวเลือกวิธีการจำลองที่ Aer รองรับได้ที่ [เอกสาร AerSimulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator)

เพื่อสำรวจการจำลองแบบแม่นยำและแบบมีสัญญาณรบกวน ให้สร้าง circuit ตัวอย่างบน 8 qubit:

In [1]:
from qiskit.circuit.library import efficient_su2

n_qubits = 8
circuit = efficient_su2(n_qubits)
circuit.draw("mpl")

<Image src="../docs/images/guides/simulate-with-qiskit-aer/extracted-outputs/df70b5fd-971d-4e7d-a23a-8df037c0fa47-0.svg" alt="Output of the previous code cell" />

![ผลลัพธ์จากโค้ดเซลล์ก่อนหน้า](../docs/images/guides/simulate-with-qiskit-aer/extracted-outputs/df70b5fd-971d-4e7d-a23a-8df037c0fa47-0.svg)

Circuit นี้มีพารามิเตอร์สำหรับแทนค่ามุมการหมุนของ Gate $R_y$ และ $R_z$ เมื่อจำลอง circuit นี้ เราต้องระบุค่าที่ชัดเจนสำหรับพารามิเตอร์เหล่านี้ ในเซลล์ถัดไป เราจะระบุค่าบางส่วนสำหรับพารามิเตอร์เหล่านี้ และใช้ Estimator primitive จาก Qiskit Aer เพื่อคำนวณค่าคาดหวังที่แม่นยำของ observable $ZZ \cdots Z$

In [2]:
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as Estimator

observable = SparsePauliOp("Z" * n_qubits)
params = [0.1] * circuit.num_parameters

exact_estimator = Estimator()
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(3, AerSimulator())
isa_circuit = pass_manager.run(circuit)
pub = (isa_circuit, observable, params)
job = exact_estimator.run([pub])
result = job.result()
pub_result = result[0]
exact_value = float(pub_result.data.evs)
exact_value

0.8870140234256602

Now, let's initialize a noise model that includes depolarizing error of 2% on every CX gate. In practice, the error arising from the two-qubit gates, which are CX gates here, are the dominant source of error when running a circuit. See [Build noise models](./build-noise-models) for an overview of constructing noise models in Qiskit Aer.

In the next cell, we construct an Estimator that incorporates this noise model and use it to compute the expectation value of the observable.

In [3]:
from qiskit_aer.noise import NoiseModel, depolarizing_error

noise_model = NoiseModel()
cx_depolarizing_prob = 0.02
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(cx_depolarizing_prob, 2), ["cx"]
)

noisy_estimator = Estimator(
    options=dict(backend_options=dict(noise_model=noise_model))
)
job = noisy_estimator.run([pub])
result = job.result()
pub_result = result[0]
noisy_value = float(pub_result.data.evs)
noisy_value

0.7247404214143528

ตอนนี้ มาเริ่มต้นแบบจำลองสัญญาณรบกวนที่รวม depolarizing error 2% บน CX gate ทุกตัวกัน ในทางปฏิบัติ ข้อผิดพลาดที่เกิดจาก two-qubit gate ซึ่งในที่นี้คือ CX gate เป็นแหล่งข้อผิดพลาดหลักเมื่อรัน circuit ดูที่ [สร้างแบบจำลองสัญญาณรบกวน](./build-noise-models) สำหรับภาพรวมของการสร้างแบบจำลองสัญญาณรบกวนใน Qiskit Aer

ในเซลล์ถัดไป เราจะสร้าง Estimator ที่รวมแบบจำลองสัญญาณรบกวนนี้ และใช้มันคำนวณค่าคาดหวังของ observable

In [4]:
cx_count = circuit.count_ops()["cx"]
(1 - cx_depolarizing_prob) ** cx_count

0.6542558123199923

This value, 65%, gives a rough estimate of the probability that our final state is correct. It is a conservative estimate because it does not take into account the initial state of the simulation.

The following code cell shows how to use the Sampler primitive from Qiskit Aer to sample from the noisy circuit. We need to add measurements to the circuit before running it with the Sampler primitive.

In [5]:
from qiskit_aer.primitives import SamplerV2 as Sampler

measured_circuit = circuit.copy()
measured_circuit.measure_all()

noisy_sampler = Sampler(
    options=dict(backend_options=dict(noise_model=noise_model))
)
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(3, AerSimulator())
isa_circuit = pass_manager.run(measured_circuit)
pub = (isa_circuit, params, 100)
job = noisy_sampler.run([pub])
result = job.result()
pub_result = result[0]
pub_result.data.meas.get_counts()

{'00100000': 1,
 '00000000': 65,
 '10101000': 1,
 '10000000': 5,
 '00001000': 1,
 '00000110': 2,
 '11110010': 1,
 '00000011': 3,
 '01010000': 3,
 '11000000': 3,
 '01111000': 1,
 '01000000': 2,
 '00000010': 1,
 '01100000': 1,
 '00011000': 1,
 '00111100': 1,
 '00010100': 1,
 '00001111': 1,
 '00110000': 1,
 '01100101': 1,
 '00000100': 1,
 '10100000': 1,
 '00000001': 1,
 '11010000': 1}

อย่างที่เห็น ค่าคาดหวังเมื่อมีสัญญาณรบกวนนั้นห่างจากค่าที่ถูกต้องค่อนข้างมาก ในทางปฏิบัติ คุณสามารถใช้เทคนิคการลดข้อผิดพลาดหลายรูปแบบเพื่อต่อสู้กับผลกระทบของสัญญาณรบกวน แต่การอภิปรายเกี่ยวกับเทคนิคเหล่านี้อยู่นอกขอบเขตของบทความนี้

เพื่อให้เห็นภาพคร่าวๆ ว่าสัญญาณรบกวนส่งผลต่อผลลัพธ์สุดท้ายอย่างไร ให้พิจารณาแบบจำลองสัญญาณรบกวนของเรา ซึ่งเพิ่ม depolarizing error 2% ให้กับแต่ละ CX gate Depolarizing error ที่มีความน่าจะเป็น $p$ ถูกกำหนดเป็น quantum channel $E$ ที่มีการกระทำต่อ density matrix $\rho$ ดังนี้:

$$
E(\rho) = (1 - p) \rho + p\frac{I}{2^n}
$$

โดยที่ $n$ คือจำนวน qubit ซึ่งในกรณีนี้คือ 2 กล่าวคือ ด้วยความน่าจะเป็น $p$ สถานะจะถูกแทนที่ด้วย completely mixed state และสถานะจะถูกรักษาไว้ด้วยความน่าจะเป็น $1 - p$ หลังจากการใช้งาน depolarizing channel $m$ ครั้ง ความน่าจะเป็นที่สถานะจะถูกรักษาไว้จะเป็น $(1 - p)^m$ ดังนั้น เราคาดว่าความน่าจะเป็นในการรักษาสถานะที่ถูกต้องไว้ตอนสิ้นสุดการจำลองจะลดลงแบบ exponential ตามจำนวน CX gate ใน circuit ของเรา

มาลองนับจำนวน CX gate ใน circuit ของเราและคำนวณ $(1 - p)^m$ กัน เราเรียก `count_ops` เพื่อรับ dictionary ที่แมป gate name ไปยังจำนวน และดึงรายการสำหรับ CX gate